In [17]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    ExtraTreesRegressor
)
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

In [2]:
# 1. Load the dataset
df = pd.read_csv("university_library_crowd.csv")

target = "people_present"

# Remove rows without a target
df = df.dropna(subset=[target]).copy()

# 2. Prepare features
# Initially exclude the raw date
X = df.drop(columns=[target, "date"], errors="ignore")
y = df[target]

# Ensure numeric target
y = pd.to_numeric(y, errors="raise")

In [16]:
print("mean:", y.mean())
print("range:", y.min(), ",", y.max())

mean: 173.757
range: 50 , 552


In [3]:
# Treat day_of_week as categorical
categorical_features = ["day_of_week"]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

In [4]:
# 3. Split before fitting any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

In [5]:
# 4. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [23]:
# 5. Candidate models
models = {
    "Linear Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]),

    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Hist Gradient Boosting": Pipeline([
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        ))
    ])
}

In [24]:
# 6. Train and evaluate
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    })

    trained_models[name] = model

In [25]:
models2 = {
    "XGBoost": Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Extra Trees": Pipeline([
        ("preprocess", preprocessor),
        ("model", ExtraTreesRegressor(
            n_estimators=300,
            max_features=1.0,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

In [26]:
# Train and evaluate

for name, model in models2.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    })

    trained_models[name] = model


In [27]:
results_df = pd.DataFrame(results).sort_values("MAE")
results_df['MAPE'] *= 100;
print("\nModel comparison:")
print(results_df.to_string(index=False))


Model comparison:
                 Model       MAE      RMSE       R2      MAPE
Hist Gradient Boosting 23.626245 30.340740 0.901993 17.847463
               XGBoost 23.801329 30.551397 0.900627 17.938907
           Extra Trees 24.438190 31.928463 0.891467 17.710578
         Random Forest 24.576175 31.800769 0.892333 18.009480
     Linear Regression 49.199023 59.574778 0.622140 41.648419


In [28]:
# 7. Save the best model by test MAE
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

joblib.dump(best_model, "crowd_prediction_model.pkl")

print(f"\nSaved model: {best_name}")
print("File: crowd_prediction_model.pkl")


Saved model: Hist Gradient Boosting
File: crowd_prediction_model.pkl
